# Education Outflow Prediction
This notebook loads the dbt-created DuckDB dataset and trains a simple prediction model using education-based migration features.

In [1]:
import duckdb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import joblib

In [3]:
con = duckdb.connect('../duckdb/brazil_migration.duckdb')

# Adjust the query below if you want to use a different dbt model or analysis output.
query = '''
select year, education_group, total_migrants, male_ratio, female_ratio
from analytics_marts.dim_brazil_education_features
'''

data = con.execute(query).df()

In [4]:
data = data.dropna(subset=['total_migrants'])
data['education_group'] = data['education_group'].astype('category').cat.codes
X = data[['education_group', 'male_ratio', 'female_ratio']]
y = data['total_migrants']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=50, random_state=42)
model.fit(X_train, y_train)
joblib.dump(model, '../models/prediction_model.pkl')

['../models/prediction_model.pkl']

## Quick prediction example
Use the trained model to predict the next total migrant count for a high-education group.

In [5]:
sample = pd.DataFrame([{'education_group': 1, 'male_ratio': 0.5, 'female_ratio': 0.5}])
print('Prediction for sample:', model.predict(sample))

Prediction for sample: [340334.76]
